# Mini-projet guidé — Classification Fashion-MNIST

**Mise en situation**

Vous êtes ingénieur(e) ML chez un site e-commerce de vêtements. On vous demande de prototyper un classifieur capable de reconnaître automatiquement la **catégorie de vêtement** sur une photo carrée en niveaux de gris (28×28, dataset Fashion-MNIST : t-shirt, pantalon, pull, robe, manteau, sandale, chemise, basket, sac, bottine).

Vous allez :

1. **Explorer** le dataset.
2. Implémenter un **baseline** : régression logistique sur pixels bruts.
3. Implémenter un **CNN** simple.
4. **Comparer** les deux approches et conclure.

**Durée indicative : 30-40 minutes** (et synthèse du module).

> On utilise un sous-ensemble (5 000 train / 1 000 test) pour la rapidité.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"
FMNIST_CLASSES = [
    "T-shirt",
    "Pantalon",
    "Pull",
    "Robe",
    "Manteau",
    "Sandale",
    "Chemise",
    "Basket",
    "Sac",
    "Bottine",
]

## Étape 1 — Exploration

1. Charge Fashion-MNIST train et test (`datasets.FashionMNIST`, `transform=transforms.ToTensor()`).
2. Restreins aux 5 000 / 1 000 premiers exemples.
3. Affiche 10 exemples (un par classe) avec le nom de classe en titre.
4. Affiche le nombre d'exemples par classe dans le train.

<details>
<summary>💡 Coup de pouce — explorer Fashion-MNIST</summary>

**🎯 But :** charger Fashion-MNIST, visualiser un exemple par classe, et vérifier l'équilibrage.

**Définir les noms de classes**

```python
FASHION_CLASSES = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
                   'Sandale',  'Chemise',  'Basket', 'Sac', 'Bottine']
```

L'ordre des labels 0-9 — fixé par les créateurs du dataset. Indispensable pour les titres lisibles dans les visualisations.

**Charger le dataset**

```python
from torchvision import datasets, transforms
transform = transforms.ToTensor()
train = datasets.FashionMNIST('./data', train=True,  download=True, transform=transform)
test  = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
```

**Restreindre à 5 000 / 1 000**

```python
from torch.utils.data import Subset
train_small = Subset(train, range(5000))
test_small  = Subset(test,  range(1000))
```

Subset garde les `Dataset` mais filtre les indices accessibles → ultra-rapide et pas de copie.

**Afficher une image par classe**

```python
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
seen = set()
for i in range(len(train_small)):
    img, label = train_small[i]
    if label not in seen:
        ax = axes.flat[label]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(FASHION_CLASSES[label], fontsize=10)
        ax.axis('off')
        seen.add(label)
    if len(seen) == 10:
        break
```

Logique : on itère le dataset, on garde la **première image rencontrée** pour chaque label, et on s'arrête dès qu'on a les 10.

**Distribution des classes**

```python
import numpy as np
y_train = np.array([train_small[i][1] for i in range(len(train_small))])
counts = np.bincount(y_train)
plt.bar(FASHION_CLASSES, counts)
plt.xticks(rotation=30)
```

`np.bincount(y)` compte les occurrences de chaque entier 0, 1, ..., 9. Devrait montrer ~500 par classe (équilibré par construction).

**Ce que vous devriez voir**
- Les hauts (T-shirt, Pull, Manteau, Chemise) sont **graphiquement très similaires** → annonceur des confusions à venir.
- Les chaussures (Sandale, Basket, Bottine) sont **proches aussi**.
- Sac, Pantalon, Robe sont plus **distinctifs**.

</details>

In [ ]:
# TODO


## Étape 2 — Baseline : régression logistique sur pixels

1. Extrais `X_train, y_train, X_test, y_test` en numpy. `X_train.shape` doit être `(5000, 784)` après aplatissement.
2. Construis un Pipeline `StandardScaler + LogisticRegression(max_iter=1000)`.
3. Fit et évalue sur le test.
4. Affiche l'accuracy et la matrice de confusion (`ConfusionMatrixDisplay.from_estimator`).

Quel score obtient-on ?

<details>
<summary>💡 Coup de pouce — baseline : régression logistique sur pixels</summary>

**🎯 But :** établir un score de référence avec un modèle simple **avant** de passer aux CNN.

**Convertir les Subsets en numpy**

```python
import numpy as np
X_train = np.array([train_small[i][0].numpy().flatten() for i in range(len(train_small))])
y_train = np.array([train_small[i][1]                  for i in range(len(train_small))])
X_test  = np.array([test_small[i][0].numpy().flatten() for i in range(len(test_small))])
y_test  = np.array([test_small[i][1]                  for i in range(len(test_small))])
print(X_train.shape)   # (5000, 784) — 28*28 aplati
```

⚠️ **`.numpy()` puis `.flatten()`** : conversion `(1, 28, 28)` → `(784,)`. Si on enlève `.flatten()`, on aurait `(5000, 1, 28, 28)` qu'un classifieur sklearn refuserait.

**Pipeline standard sklearn**

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000)),
])
baseline.fit(X_train, y_train)
acc = baseline.score(X_test, y_test)
print(f"Baseline accuracy : {acc:.3f}")
```

**Pourquoi `max_iter=1000`** ?

Sur 5000 × 784 features, LogReg peut être lente à converger. `max_iter=1000` (au lieu du 100 par défaut) évite le warning « ConvergenceWarning ». La perf finale est identique.

**Résultats attendus**

Sur Fashion-MNIST avec 5000 exemples :
- **Baseline LogReg ≈ 80-85 %**
- C'est le **plancher** : tout modèle plus complexe devra battre ça pour justifier sa complexité.

**Matrice de confusion**

```python
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_estimator(
    baseline, X_test, y_test, display_labels=FASHION_CLASSES, ax=ax, xticks_rotation=45
)
```

Vous verrez les confusions classiques :
- Hauts entre eux (T-shirt/Pull/Chemise/Manteau)
- Chaussures entre elles (Sandale/Basket/Bottine)
- Sac et Pantalon généralement distincts.

</details>

In [ ]:
# TODO


## Étape 3 — CNN simple

Réutilise (ou recopie) l'architecture `SmallCNN` du TP 1 (2 blocs convolutifs).

Entraîne sur Fashion-MNIST 3 époques avec Adam(lr=1e-3) et `CrossEntropyLoss`.

Affiche la courbe d'accuracy train / test. Compare avec la baseline.

<details>
<summary>💡 Coup de pouce — CNN simple sur Fashion-MNIST</summary>

**🎯 But :** appliquer l'architecture vue en TP1 MNIST sur Fashion-MNIST (même format), comparer avec la baseline.

**Le réflexe « même problème, même architecture »**

Fashion-MNIST est techniquement identique à MNIST :
- 28×28 mono-canal
- 10 classes
- 60 000 train / 10 000 test

→ Vous pouvez **reprendre exactement le SmallCNN du TP1 (CNN MNIST)**, copié-collé, et il fonctionnera. La seule différence est la **difficulté** (Fashion-MNIST est plus dur).

**Recopier l'architecture**

```python
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)
```

**Setup entraînement**

```python
model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(train_small, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_small,  batch_size=256, shuffle=False)
```

**Boucle d'entraînement avec courbe train/test**

```python
train_accs, test_accs = [], []
for epoch in range(3):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    # éval après l'époque
    def evaluate(loader):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        return correct / total

    train_accs.append(evaluate(train_loader))
    test_accs.append(evaluate(test_loader))
    print(f"Epoch {epoch+1}: train={train_accs[-1]:.3f}  test={test_accs[-1]:.3f}")

# Tracer
plt.plot(train_accs, marker='o', label='train')
plt.plot(test_accs,  marker='s', label='test')
plt.xlabel('Époque'); plt.ylabel('Accuracy'); plt.legend()
```

**Résultats attendus**

- **Baseline LogReg ≈ 80-85 %**
- **CNN simple ≈ 88-92 %**

Gain de **+5 à +10 points** vs baseline. C'est typique : un CNN bat un modèle linéaire sur images, mais pas démesurément sur Fashion-MNIST (qui reste un dataset relativement simple).

</details>

In [ ]:
# TODO


## Étape 4 — Synthèse

Dans la cellule markdown suivante, rédige (en 5 phrases environ) :

1. L'accuracy obtenue par la baseline et par le CNN.
2. Quelles classes sont les plus confondues ? (regarde la matrice de confusion du CNN).
3. Comment irais-tu chercher 1-2 points d'accuracy supplémentaires (idées concrètes : augmentation, profondeur, transfer learning, plus de données…).

<details>
<summary>💡 Coup de pouce — synthèse et pistes d'amélioration</summary>

**🎯 But :** rédiger une analyse concise des résultats et proposer des pistes concrètes d'amélioration.

**Structure de la synthèse**

1. **Quels scores ?** Baseline X %, CNN Y %, gain Z points.
2. **Quelles classes confondues ?** Matrice de confusion du CNN — paires les plus fréquentes.
3. **Pistes pour +1-2 pts** — choix techniques avec justifications.

**Confusions typiques sur Fashion-MNIST**

| Paire confondue | Pourquoi |
|---|---|
| Chemise ↔ T-shirt ↔ Pull | Tous des hauts, contours similaires |
| Bottine ↔ Sandale ↔ Basket | Toutes des chaussures, silhouettes proches |
| Manteau vs Pull | Différenciation difficile sans couleur |

Faites une analyse au cas par cas en regardant **les images mal classées** (comme en TP1 MNIST).

**Pistes d'amélioration — par ordre d'effort / impact**

| Piste | Effort | Gain attendu |
|---|---|---|
| **Plus de données** (utiliser les 60k au lieu de 5k) | Faible | +3-5 pts |
| **Augmentation** (flip horizontal, crop) | Faible | +1-2 pts |
| **CNN plus profond** (3 blocs au lieu de 2) | Moyen | +1-2 pts |
| **BatchNorm + Dropout** dans le CNN | Faible | +1 pt |
| **Transfer learning** (ResNet18, MobileNet pré-entraîné) | Moyen | +3-5 pts |
| **Ensembles** (moyenne de plusieurs modèles) | Élevé | +1-2 pts |
| **Régularisation Mixup / CutOut** | Moyen | +1-2 pts |

**Compromis budget de calcul**

Sur 5 000 exemples, le gain marginal des techniques avancées (transfer, ensembles) est moins évident que sur datasets plus gros. À petit budget, la **donnée et l'augmentation** restent les leviers les plus rentables.

**Exemple de rédaction**

> La baseline LogReg atteint 83 % sur 5000 exemples ; le CNN simple monte à 89 %, soit +6 points. La matrice de confusion révèle que les confusions majeures sont entre Chemise/T-shirt/Pull (catégorie « hauts ») et entre Bottine/Sandale/Basket (catégorie « chaussures »). Pour gagner 1-2 points supplémentaires, je commencerais par augmenter le dataset à 60k examples (gain attendu : +3 pts), ajouter une augmentation horizontale (+1 pt), puis tester un transfer learning ResNet18 pré-entraîné sur ImageNet (+2-3 pts). Sur ce dataset, augmenter la profondeur du CNN sans plus de données risque l'overfitting.

</details>

*Ta synthèse ici*